# 지연 위험도(정상/주의/위험) 분류 모델 — LightGBM

피처 선정 → 이슈 단위 층화 분할 → LightGBM multiclass 학습 → 평가 → 저장까지의 학습 파이프라인을
함수로 감싸지 않고, 각 단계를 셀 단위로 LightGBM/scikit-learn 함수를 직접 호출하며 실행합니다.
중간 결과(피처 후보/딕셔너리, 분할 결과, 클래스 분포, 피처 중요도, 혼동 행렬 등)는 각 단계 직후 셀에서
바로 확인할 수 있습니다.

다만 모델 저장/로드(`load_artifact`)와 실시간 추론(`predict_class_probabilities`, `proxy_deadline_for`)은
`App/backend_fastapi/ml_delay_risk/services/delay_service.py`가 `_notebook_runtime.load()`를 통해
이 노트북에서 직접 가져다 쓰므로 함수 형태로 유지합니다.

실제로 이 노트북에서 학습을 실행하려면 `ml_delay_risk/config.py`에 설정된 MongoDB(`ml_dashboard`)에
접속 가능해야 합니다.


# Feature Descriptions (피처 설명 목록)

| 피처명 (Feature Name) | 설명 (Description) |
| :--- | :--- |
| `project_key` | 프로젝트 코드 (이슈 키 접두사, 예: JELLY) |
| `issuetype_name` | 이슈 유형 (Bug/Task/Improvement 등) |
| `priority_name` | 우선순위 (Blocker/Critical/Major/Minor/Trivial) |
| `reporter` | 이슈 등록자 (빈도 인코딩) |
| `is_subtask` | 하위 작업(subtask) 여부 |
| `has_parent` | 상위 이슈(에픽 등)에 속해 있는지 여부 |
| `parent_unresolved` | 상위 이슈(에픽)가 아직 미해결 상태인지 여부 |
| `num_subtasks` | 하위 작업 개수 |
| `num_unresolved_subtasks` | 하위 작업 중 미해결 개수 |
| `num_components` | 소속 컴포넌트(모듈) 개수 |
| `num_fixversions` | 목표 릴리즈 버전 개수 |
| `has_released_fixversion` | 목표 릴리즈 버전 중 이미 배포된 것이 있는지 여부 |
| `num_versions` | 영향받는 버전 개수 |
| `has_original_estimate` | 최초 예상 소요시간(estimate)이 설정되어 있는지 여부 |
| `original_estimate_seconds` | 최초 예상 소요시간(초 단위) |
| `num_issuelinks_total` | 다른 이슈와의 연관관계 총 개수 |
| `num_blocked_by_links` | '~에 의해 막힘' 관계 개수 |
| `num_unresolved_blockers` | 블로커 이슈 중 아직 해결되지 않은 것의 개수 |
| `created_day_of_week` | 생성 요일 (0=월요일 ~ 6=일요일) |
| `created_hour` | 생성 시각 (0~23시) |
| `summary_length` | 이슈 제목 길이 (문자 수, 복잡도 근사치) |
| `status_at_cutoff` | cutoff 시점의 상태 (Open/In Progress/Blocked 등) |
| `assignee_at_cutoff` | cutoff 시점의 담당자 (빈도 인코딩) |
| `num_events_before_cutoff` | cutoff 이전 변경 이력(이벤트) 총 개수 |
| `num_status_changes` | 상태 변경 횟수 |
| `num_assignee_changes` | 담당자 변경(재배정) 횟수 |
| `num_reopens` | 상태가 Open/Reopened로 되돌아간 횟수 |
| `hours_in_current_status` | 마지막 상태 변경 이후 ~ cutoff까지 경과한 시간 |
| `blocked_hours_before_cutoff` | 블로커 상태에 머문 누적 시간 |
| `num_comments_before_cutoff` | cutoff 이전 댓글 수 |
| `num_unique_commenters` | 댓글을 남긴 고유 인원 수 |
| `hours_since_last_comment` | 마지막 댓글로부터 ~ cutoff까지 경과 시간 |
| `num_worklog_entries` | 작업 기록(worklog) 건수 (봇 계정 제외) |
| `num_unique_workers` | 실제 작업한 고유 인원 수 |
| `time_spent_seconds_before_cutoff` | cutoff까지 누적 투입 시간 (초, 봇 제외) |
| `progress_ratio_at_cutoff` | 진행률 (누적 투입시간 ÷ 최초 예상시간) |
| `elapsed_hours_at_cutoff` | 이슈 생성 후 ~ cutoff까지 경과 시간 (절대값, 시간 단위) |
| `activity_count_recent_window` | 최근 N일간 활동(댓글+상태변경+작업기록) 합계 |
| `is_self_assigned` | 보고자와 담당자가 동일 인물인지 여부 |
| `snapshot_offset_days` | 스냅샷 시점 (이슈 생성 후 며칠째인지) |

## 1. 환경 설정 — 노트북 위치와 무관하게 backend_fastapi를 찾아 `sys.path`에 추가

In [154]:
import sys
from pathlib import Path


def _find_backend_fastapi_root() -> Path:
    # VS Code Jupyter 확장은 현재 열린 노트북의 절대경로를 __vsc_ipynb_file__ 전역변수로 제공한다.
    # 이게 없는 환경(JupyterLab 등)에서는 커널의 현재 작업 디렉터리(cwd)로 대체한다.
    notebook_path = globals().get("__vsc_ipynb_file__")
    search_start = Path(notebook_path).resolve().parent if notebook_path else Path.cwd()

    # 노트북이 ml_delay_risk/ 바로 아래에 있든 models/ 등 하위 폴더로
    # 옮겨지든 상관없이 상위 폴더를 훑어 backend_fastapi(=ml_delay_risk의
    # 부모)를 찾는다. cwd에 의존하는 방식(Path.cwd().parent)은 노트북 프론트엔드마다
    # 커널 작업 디렉터리가 달라 깨지기 쉬워 사용하지 않는다.
    for candidate in [search_start, *search_start.parents]:
        if (candidate / "ml_delay_risk").is_dir():
            return candidate
        if (candidate / "backend_fastapi" / "ml_delay_risk").is_dir():
            return candidate / "backend_fastapi"

    raise RuntimeError(
        f"'{search_start}' 및 상위 폴더에서 ml_delay_risk 패키지를 찾지 못했습니다.\n"
        "노트북이 backend_fastapi 프로젝트 안에 있는지 확인하거나, 아래처럼 경로를 직접 추가하세요:\n"
        '    sys.path.insert(0, r"D:\\...\\backend_fastapi")'
    )


BACKEND_FASTAPI_ROOT = _find_backend_fastapi_root()
if str(BACKEND_FASTAPI_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_FASTAPI_ROOT))

print("BACKEND_FASTAPI_ROOT =", BACKEND_FASTAPI_ROOT)
print("sys.path[0] =", sys.path[0])


BACKEND_FASTAPI_ROOT = D:\AIproject\project\Team\work-flow\App\backend_fastapi
sys.path[0] = D:\AIproject\project\Team\work-flow\App\backend_fastapi


## 2. Imports

In [155]:
from __future__ import annotations

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# 설치된 폰트 중 한글 폰트가 있을 때만 지정한다. 리스트로 지정하면 matplotlib가
# 글자 단위로 각 후보 폰트를 확인하며 없는 폰트마다 findfont 경고를 반복 출력하므로,
# font_manager로 실제 설치 여부를 미리 확인해 존재하는 폰트 하나만 설정한다.
# 아무 것도 없으면(Docker/Linux 등) 건드리지 않고 matplotlib 기본 폰트를 그대로 쓴다
# (한글 라벨은 tofu로 깨져 보일 수 있지만 경고는 나지 않는다).
import matplotlib.font_manager as fm

_installed_fonts = {f.name for f in fm.fontManager.ttflist}
_korean_font = next(
    (name for name in ("Malgun Gothic", "NanumGothic", "AppleGothic") if name in _installed_fonts),
    None,
)
if _korean_font:
    plt.rcParams["font.family"] = _korean_font
plt.rcParams["axes.unicode_minus"] = False  # 위 폰트 사용 시 마이너스 기호 깨짐 방지
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from ml_delay_risk.models.feature_engineering import RISK_CLASS_NAMES
from ml_delay_risk.config import get_settings

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


## 3. 상수 정의

- `CATEGORICAL_COLUMNS`: LightGBM 네이티브 범주형 처리 대상 (저카디널리티).
- `FREQUENCY_ENCODED_COLUMNS`: 담당자/보고자처럼 카디널리티가 높은 컬럼은 빈도 인코딩으로 처리.
- `NON_FEATURE_COLUMNS`: 학습에 쓰지 않는 부기(bookkeeping) 컬럼. `proxy_deadline_hours`는
  issuetype/priority로 이미 결정되는 중복 정보라 피처에서 제외합니다.


In [156]:
NUM_CLASSES = 3
CATEGORICAL_COLUMNS = ["issuetype_name", "priority_name", "project_key", "status_at_cutoff"]
FREQUENCY_ENCODED_COLUMNS = ["reporter", "assignee_at_cutoff"]

# <데이터 누수 방지>
# (해결)
# LEAKY_FEATURE_COLUMNS를 NON_FEATURE_COLUMNS에 재합류 (cell 6) 
# — elapsed_ratio_at_cutoff, blocked_ratio_at_cutoff, imbalance_index_at_cutoff, hours_until_deadline_at_cutoff 
# 4개를 다시 피처에서 제외.

# (문제 원인)
# elapsed_ratio_at_cutoff / blocked_ratio_at_cutoff / imbalance_index_at_cutoff는
# risk_class 라벨을 계산하는 classify_risk()의 입력값 그 자체이고,
# hours_until_deadline_at_cutoff는 proxy_deadline_hours - elapsed_hours_at_cutoff로
# 역산 가능해 사실상 동일한 정보다. 피처로 남겨두면 모델이 패턴을 학습하는 대신
# 라벨링 규칙식을 그대로 베껴버린다(타겟 누수).
LEAKY_FEATURE_COLUMNS = [
    "elapsed_ratio_at_cutoff",
    "blocked_ratio_at_cutoff",
    "imbalance_index_at_cutoff",
    "hours_until_deadline_at_cutoff",
]
NON_FEATURE_COLUMNS = [
    "issue_key",
    "created",
    "risk_class",
    "proxy_deadline_hours",
] + LEAKY_FEATURE_COLUMNS

# 후보 피처들의 의미 설명 (선정 결과를 사람이 읽을 수 있게 출력할 때 사용).
# feature_engineering.py의 build_static_features/build_dynamic_features가 만드는
# 필드와 1:1로 대응한다.
FEATURE_DESCRIPTIONS: dict[str, str] = {
    "project_key": "프로젝트 코드 (이슈 키 접두사, 예: JELLY)",
    "issuetype_name": "이슈 유형 (Bug/Task/Improvement 등)",
    "priority_name": "우선순위 (Blocker/Critical/Major/Minor/Trivial)",
    "reporter": "이슈 등록자 (빈도 인코딩)",
    "is_subtask": "하위 작업(subtask) 여부",
    "has_parent": "상위 이슈(에픽 등)에 속해 있는지 여부",
    "parent_unresolved": "상위 이슈(에픽)가 아직 미해결 상태인지 여부",
    "num_subtasks": "하위 작업 개수",
    "num_unresolved_subtasks": "하위 작업 중 미해결 개수",
    "num_components": "소속 컴포넌트(모듈) 개수",
    "num_fixversions": "목표 릴리즈 버전 개수",
    "has_released_fixversion": "목표 릴리즈 버전 중 이미 배포된 것이 있는지 여부",
    "num_versions": "영향받는 버전 개수",
    "has_original_estimate": "최초 예상 소요시간(estimate)이 설정되어 있는지 여부",
    "original_estimate_seconds": "최초 예상 소요시간(초 단위)",
    "num_issuelinks_total": "다른 이슈와의 연관관계 총 개수",
    "num_blocked_by_links": "'~에 의해 막힘' 관계 개수",
    "num_unresolved_blockers": "블로커 이슈 중 아직 해결되지 않은 것의 개수",
    "created_day_of_week": "생성 요일 (0=월요일 ~ 6=일요일)",
    "created_hour": "생성 시각 (0~23시)",
    "summary_length": "이슈 제목 길이 (문자 수, 복잡도 근사치)",
    "status_at_cutoff": "cutoff 시점의 상태 (Open/In Progress/Blocked 등)",
    "assignee_at_cutoff": "cutoff 시점의 담당자 (빈도 인코딩)",
    "num_events_before_cutoff": "cutoff 이전 변경 이력(이벤트) 총 개수",
    "num_status_changes": "상태 변경 횟수",
    "num_assignee_changes": "담당자 변경(재배정) 횟수",
    "num_reopens": "상태가 Open/Reopened로 되돌아간 횟수",
    "hours_in_current_status": "마지막 상태 변경 이후 ~ cutoff까지 경과한 시간",
    "blocked_hours_before_cutoff": "블로커 상태에 머문 누적 시간",
    "num_comments_before_cutoff": "cutoff 이전 댓글 수",
    "num_unique_commenters": "댓글을 남긴 고유 인원 수",
    "hours_since_last_comment": "마지막 댓글로부터 ~ cutoff까지 경과 시간",
    "num_worklog_entries": "작업 기록(worklog) 건수 (봇 계정 제외)",
    "num_unique_workers": "실제 작업한 고유 인원 수",
    "time_spent_seconds_before_cutoff": "cutoff까지 누적 투입 시간 (초, 봇 제외)",
    "progress_ratio_at_cutoff": "진행률 (누적 투입시간 ÷ 최초 예상시간)",
    "elapsed_hours_at_cutoff": "이슈 생성 후 ~ cutoff까지 경과 시간 (절대값, 시간 단위)",
    "activity_count_recent_window": "최근 N일간 활동(댓글+상태변경+작업기록) 합계",
    "is_self_assigned": "보고자와 담당자가 동일 인물인지 여부",
    "snapshot_offset_days": "스냅샷 시점 (이슈 생성 후 며칠째인지)",
}


## 3b. `_RUN_TRAINING_CELLS` — 학습/EDA 셀 실행 여부 플래그

아래부터 등장하는 `if _RUN_TRAINING_CELLS:` 블록(학습 실행, 피처 중요도 시각화, 혼동 행렬 등)은
노트북을 직접 열어 대화형으로 실행할 때만 돌아야 하고, `_notebook_runtime.load()`가
`load_artifact`/`predict_class_probabilities` 등 함수만 가져다 쓸 때는 실행되면 안 된다.

Jupyter는 노트북 커널의 `__name__`을 기본적으로 `"__main__"`으로 설정하므로, 아래 셀에서
직접 실행 여부를 판별한다. `_notebook_runtime.load(run_main=True)`(예: `train.py`)가
이 값을 미리 `True`로 주입해 준 경우에는 그 값을 그대로 쓴다.


In [ ]:
_RUN_TRAINING_CELLS = globals().get("_RUN_TRAINING_CELLS", __name__ == "__main__")


In [ ]:
if _RUN_TRAINING_CELLS:
    print(f"FEATURE_DESCRIPTIONS 딕셔너리 — 총 {len(FEATURE_DESCRIPTIONS)}개 컬럼 설명")
    display(pd.DataFrame(FEATURE_DESCRIPTIONS.items(), columns=["feature", "description"]))


## 4. 모델 아티팩트 정의

학습된 booster와 함께, 추론 시 동일하게 재현해야 하는 인코딩 맵·Proxy Deadline 조회 테이블을 함께 묶어 저장합니다.

In [158]:
@dataclass
class ModelArtifact:
    booster: lgb.Booster
    feature_names: list[str]
    categorical_columns: list[str]
    frequency_maps: dict[str, dict[str, int]]
    proxy_deadline_map: dict[tuple[str, str], float]
    global_median_duration_hours: float


## 5. 모델 저장/로드 함수

학습된 booster를 파일로 저장/로드합니다. `load_artifact`는 아래 6번 실시간 추론 함수가 바로 사용하고,
`App/backend_fastapi/ml_delay_risk/services/delay_service.py`가 `_notebook_runtime.load()`를 통해
이 노트북에서 직접 가져다 쓰므로, 다른 학습 파이프라인 코드와 달리 함수 형태를 유지합니다.

In [159]:
def _model_path() -> Path:
    settings = get_settings()
    path = Path(settings.model_dir)
    path.mkdir(parents=True, exist_ok=True)
    return path / settings.model_filename


def _save_artifact(artifact: ModelArtifact) -> None:
    path = _model_path()
    joblib.dump(artifact, path)
    logger.info("모델 저장 완료: %s", path)


_artifact_cache: Optional[ModelArtifact] = None


def load_artifact() -> ModelArtifact:
    global _artifact_cache
    if _artifact_cache is None:
        path = _model_path()
        if not path.exists():
            raise FileNotFoundError(
                f"학습된 모델이 없습니다: {path}. "
                "먼저 아래 '학습 실행' 셀을 실행하세요."
            )
        _artifact_cache = joblib.load(path)
    return _artifact_cache


## 6. 실시간 추론용 함수

`predict_class_probabilities`, `proxy_deadline_for`는 FastAPI 라우터(`routers/delay_router.py`
→ `services/delay_service.py`)가 `_notebook_runtime.load()`로 이 노트북에서 직접 가져다 쓰는
실시간 추론 API이므로, 함수 형태를 유지합니다.

In [160]:
def proxy_deadline_for(issuetype_name: str, priority_name: str) -> float:
    artifact = load_artifact()
    return artifact.proxy_deadline_map.get(
        (issuetype_name, priority_name), artifact.global_median_duration_hours
    )


def predict_class_probabilities(feature_row: dict[str, Any]) -> list[float]:
    """클래스 색인(0=정상,1=주의,2=위험) 순서의 확률 리스트를 반환."""
    artifact = load_artifact()
    row_df = pd.DataFrame([feature_row])

    for col in FREQUENCY_ENCODED_COLUMNS:
        if col in row_df.columns:
            row_df[col] = row_df[col].map(artifact.frequency_maps.get(col, {})).fillna(0).astype(int)

    for col in artifact.feature_names:
        if col not in row_df.columns:
            row_df[col] = None
    row_df = row_df[artifact.feature_names]

    for col in artifact.feature_names:
        if col in artifact.categorical_columns:
            row_df[col] = row_df[col].astype("category")
        else:
            # 단일 행 DataFrame에서 None(예: 예상시간 없는 이슈의 progress_ratio)은
            # object dtype이 되어 LightGBM predict가 거부한다 -> 명시적으로 float로 강제.
            row_df[col] = pd.to_numeric(row_df[col], errors="coerce")

    probabilities = artifact.booster.predict(row_df, num_iteration=artifact.booster.best_iteration)[0]
    return [float(p) for p in probabilities]


## 7. 학습 실행 — 피처 선정부터 모델 저장까지

MongoDB(`ml_dashboard`)에서 실제 학습 데이터를 구성한 뒤, 아래 셀들이 피처 중요도 분석 → 이슈 단위
층화 분할 → LightGBM 학습 → 평가 → 모델 저장까지의 전 과정을 각 단계별로 LightGBM/scikit-learn
함수를 셀에서 직접 호출하며 순서대로 실행합니다.

`dataset_builder.build_training_dataframe`은 `ml_delay_risk/models/dataset_builder.py`에
정의된, 이 노트북과 동일한 파이프라인이 사용하는 함수입니다.

`limit` 파라미터로 이슈 수를 제한하면 빠르게 파일럿 실행을 해볼 수 있습니다 (전체 데이터는 약 77만 건
규모라 시간이 오래 걸릴 수 있습니다). 운영에서는 노트북 대신 `python -m ml_delay_risk.train`을 사용하세요.

In [ ]:
"""
[target]업무 지연 위험도 3단계:
    Class 0 (정상): 마감일(Proxy Deadline) 이내에서 순항 중.
    Class 1 (주의): 마감일 이내지만 진행률이 저조하거나 블로커 상태에 짧게 머묾.
    Class 2 (위험): 마감일을 초과했거나, 블로커 상태에 장기간 머물러 활동이 정지(Silent).
"""
# <주피터 노트북 모듈화 - _notebook_runtime.py>
# _RUN_TRAINING_CELLS: 대화형 Jupyter에서 직접 실행할 때는 True(아래 새로 추가된 셀에서
# __name__ == "__main__"으로 판정), _notebook_runtime.load()가 함수 정의부만 가져다 쓸
# 때는 False. 학습/추론/탐색처럼 직접 실행되는 셀을 if _RUN_TRAINING_CELLS: 블록 안에 넣는다.
# (예전엔 __name__ == "__main__"을 직접 검사했지만, ModelArtifact 같은 클래스가 같은
# 노트북에서 정의되면 그 클래스의 __module__도 "__main__"이 되어 버려 joblib으로 저장한
# 모델을 다른 프로세스(FastAPI 서버)에서 로드할 때 피클이 깨지는 문제가 있어 이 플래그로 바꿨다.)
if _RUN_TRAINING_CELLS:
    from ml_delay_risk.models.dataset_builder import build_training_dataframe

    # <학습 데이터 갯수 증가>
    # (해결) 파일럿 limit을 300 → 1500으로 재확대
    # (이유) 희귀 클래스('주의', class 1)를 학습에 충분히 사용함

    # 파일럿 실행 예시 (전체 데이터로 학습하려면 limit=None). train.py가
    # `_TRAIN_LIMIT`을 미리 넣어주면 그 값을 쓰고, 노트북에서 직접 실행할 땐 1500이 기본값.
    limit = globals().get("_TRAIN_LIMIT", 1500)
    df, proxy_deadline_map, global_median = build_training_dataframe(limit=limit)
    if df.empty:
        raise SystemExit(
            "학습 데이터가 비어 있습니다. MongoDB(ml_dashboard) 연결/컬렉션을 확인하세요."
        )

In [ ]:
if _RUN_TRAINING_CELLS:
    # <임포트/생성 직후 데이터셋 시각화>
    # 피처 선정으로 축소되기 이전, 임포트 직후의 전체 후보 피처와 타겟(risk_class)을 확인한다.
    print("데이터셋 생성 완료 — shape:", df.shape)
    display(df.head())
    display(df.dtypes.rename("dtype"))


In [ ]:
if _RUN_TRAINING_CELLS:
    df["risk_class"].value_counts().sort_index()


In [ ]:
if _RUN_TRAINING_CELLS:
    # 시간(created) 기준 정렬 — 이후 이슈 단위 분할과 무관하게, 데이터 확인 시 시간 순서를 보장.
    df = df.sort_values("created").reset_index(drop=True)

    # 학습에 쓰지 않는 부기 컬럼(NON_FEATURE_COLUMNS)을 제외한 나머지가 피처 후보.
    candidate_features = [c for c in df.columns if c not in NON_FEATURE_COLUMNS]
    print(f"피처 후보 {len(candidate_features)}개: {candidate_features}")


In [ ]:
if _RUN_TRAINING_CELLS:
    # <컬럼을 딕셔너리 형태로 담음 — freq_maps>
    # 피처 중요도 프로브용으로, 전체 데이터 기준 "값 -> 등장 횟수" 딕셔너리를 만든다.
    freq_maps = {
        col: df[col].value_counts().to_dict()
        for col in FREQUENCY_ENCODED_COLUMNS
        if col in df.columns
    }
    print(f"빈도 인코딩 딕셔너리(freq_maps) 생성 완료 — 대상 컬럼: {list(freq_maps.keys())}")
    for col, mapping in freq_maps.items():
        print(f"  · {col}: 고유값 {len(mapping)}개 (상위 5개: {dict(list(mapping.items())[:5])})")


In [ ]:
if _RUN_TRAINING_CELLS:
    # 피처 중요도만 산정할 프로브용 인코딩 — 실제 학습(train_df/test_df)과는 별개로,
    # 전체 데이터를 그대로 인코딩해 LightGBM에 넣는다.
    encoded_df = df.copy()
    for col in FREQUENCY_ENCODED_COLUMNS:
        if col in encoded_df.columns:
            encoded_df[col] = encoded_df[col].map(freq_maps.get(col, {})).fillna(0).astype(int)
    for col in CATEGORICAL_COLUMNS:
        if col in encoded_df.columns:
            encoded_df[col] = encoded_df[col].astype("category")

    # 클래스 불균형('위험' 표본이 '정상'보다 현저히 적음) 보정용 역빈도 가중치.
    class_counts = encoded_df["risk_class"].value_counts()
    weight_map = {cls: len(encoded_df) / (len(class_counts) * count) for cls, count in class_counts.items()}
    probe_weights = encoded_df["risk_class"].map(weight_map)

    probe_categorical_columns = [c for c in CATEGORICAL_COLUMNS if c in candidate_features]
    probe_set = lgb.Dataset(
        encoded_df[candidate_features],
        label=encoded_df["risk_class"],
        weight=probe_weights,
        categorical_feature=probe_categorical_columns,
    )
    probe_booster = lgb.train(
        {
            "objective": "multiclass",
            "num_class": NUM_CLASSES,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "verbose": -1,
        },
        probe_set,
        num_boost_round=200,
    )


In [ ]:
if _RUN_TRAINING_CELLS:
    # <피처 선정 — 학습/검증 분할 이전에 전체 데이터로 수행>
    # 정규화 중요도(gain)가 importance_threshold 이상인 피처만 이후 학습에 사용한다.
    gain = probe_booster.feature_importance(importance_type="gain")
    split_count = probe_booster.feature_importance(importance_type="split")
    gain_pct = gain / gain.sum() * 100 if gain.sum() > 0 else gain

    feature_importance_report = pd.DataFrame(
        {"feature": candidate_features, "importance_gain_pct": gain_pct, "importance_split": split_count}
    ).sort_values("importance_gain_pct", ascending=False).reset_index(drop=True)

    importance_threshold = 0.01
    selected_features = feature_importance_report.loc[
        feature_importance_report["importance_gain_pct"] / 100 >= importance_threshold, "feature"
    ].tolist()

    print(
        f"피처 선정: 후보 {len(candidate_features)}개 중 {len(selected_features)}개 선정 "
        f"(기준: 중요도 {importance_threshold * 100:.1f}% 이상)"
    )
    display(feature_importance_report)


In [ ]:
if _RUN_TRAINING_CELLS:
    # <Feature importance(Gain) 시각화>
    # 선정된 피처가 실제로 무엇을 의미하는지 사람이 바로 확인할 수 있도록 설명을 함께 출력.
    print(f"\n선정된 피처 {len(selected_features)}개 설명 (중요도 내림차순):")
    selected_report = feature_importance_report[feature_importance_report["feature"].isin(selected_features)]
    for _, row in selected_report.iterrows():
        description = FEATURE_DESCRIPTIONS.get(row["feature"], "(설명 미등록)")
        print(f"  - {row['feature']} (중요도 {row['importance_gain_pct']:.1f}%): {description}")

    plot_df = selected_report.sort_values("importance_gain_pct")
    fig, ax = plt.subplots(figsize=(7, max(4, len(selected_features) * 0.35)))
    sns.barplot(data=plot_df, x="importance_gain_pct", y="feature", color="#4C72B0", ax=ax)
    ax.set_xlabel("중요도 (Gain, %)")
    ax.set_ylabel("")
    ax.set_title("선정된 피처 중요도")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
if _RUN_TRAINING_CELLS:
    # <학습/테스트 데이터 - 층화추출 적용>
    # 같은 issue_key의 여러 스냅샷이 train/valid 양쪽에 걸쳐 들어가면 그룹 누수가 생기므로,
    # 이슈 단위(issue_key)로 그룹을 통째로 한쪽에만 배정한다. 그룹 대표 라벨은 해당 이슈가
    # 도달한 최고 위험도(risk_class 최댓값)로 잡아, StratifiedGroupKFold가 희귀 클래스('주의')도
    # train/valid 양쪽에 고르게 배분하도록 한다.
    # train.py가 `_TRAIN_TEST_SIZE`를 미리 넣어주면 그 값을 쓰고,
    # 노트북에서 직접 실행할 땐 0.2가 기본값.
    test_size = globals().get("_TRAIN_TEST_SIZE", 0.2)
    group_labels = df.groupby("issue_key")["risk_class"].max()
    issue_keys = group_labels.index.to_numpy()
    n_splits = max(2, round(1 / test_size))
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_issue_idx, valid_issue_idx = next(
        splitter.split(np.zeros(len(issue_keys)), group_labels.to_numpy(), groups=issue_keys)
    )
    train_issues = set(issue_keys[train_issue_idx])
    valid_issues = set(issue_keys[valid_issue_idx])

    train_df = df[df["issue_key"].isin(train_issues)].copy()
    test_df = df[df["issue_key"].isin(valid_issues)].copy()

    # <데이터 분할 결과 시각화>
    print(
        f"학습/검증 분할 완료 — train: {len(train_df)}행 ({train_df['issue_key'].nunique()}개 이슈), "
        f"valid: {len(test_df)}행 ({test_df['issue_key'].nunique()}개 이슈)"
    )
    display(
        pd.DataFrame(
            {
                "train": train_df["risk_class"].value_counts().sort_index(),
                "valid": test_df["risk_class"].value_counts().sort_index(),
            }
        ).fillna(0).astype(int)
    )

    all_classes = set(df["risk_class"].unique())
    missing_in_train = all_classes - set(train_df["risk_class"].unique())
    missing_in_valid = all_classes - set(test_df["risk_class"].unique())
    if missing_in_train:
        logger.warning("train 세트에 없는 클래스: %s", missing_in_train)
    if missing_in_valid:
        logger.warning("valid 세트에 없는 클래스: %s", missing_in_valid)


In [ ]:
if _RUN_TRAINING_CELLS:
    # <컬럼을 딕셔너리 형태로 담음 — frequency_maps>
    # 실제 학습(train_df) 기준으로 만든 빈도 인코딩 딕셔너리.
    frequency_maps = {
        col: train_df[col].value_counts().to_dict()
        for col in FREQUENCY_ENCODED_COLUMNS
        if col in train_df.columns
    }
    print(f"빈도 인코딩 딕셔너리(frequency_maps) 생성 완료 — 대상 컬럼: {list(frequency_maps.keys())}")
    for col, mapping in frequency_maps.items():
        print(f"  · {col}: 고유값 {len(mapping)}개 (상위 5개: {dict(list(mapping.items())[:5])})")

    # train/valid 양쪽에 동일한 frequency_maps로 빈도 인코딩 + 범주형 dtype 적용.
    for target_df in (train_df, test_df):
        for col in FREQUENCY_ENCODED_COLUMNS:
            if col in target_df.columns:
                target_df[col] = target_df[col].map(frequency_maps.get(col, {})).fillna(0).astype(int)
        for col in CATEGORICAL_COLUMNS:
            if col in target_df.columns:
                target_df[col] = target_df[col].astype("category")


In [ ]:
if _RUN_TRAINING_CELLS:
    # 위에서 분할 이전 전체 데이터로 선정한 피처만 사용(피처 개수 축소).
    feature_names = [c for c in selected_features if c in train_df.columns]
    categorical_columns = [c for c in CATEGORICAL_COLUMNS if c in feature_names]

    # <사용 피처 + 타겟 결합 데이터프레임>
    # 실제 학습에 들어가는 피처(feature_names)와 타겟(risk_class)만 모아
    # 최종적으로 모델이 무엇을 보고 무엇을 맞추는지 한눈에 확인한다.
    print("<사용 피처 + 타겟 결합 데이터프레임>")
    display(df[feature_names + ["risk_class"]])


In [ ]:
if _RUN_TRAINING_CELLS:
    # <불균형 데이터 처리 - SMOTE기반 균형 샘플링>
    # 참고: https://jaylala.tistory.com/entry/불균형데이터처리-오버샘플링Oversampling-SMOTE
    #
    # 바로 아래(다음 셀)에서는 원래 lgb.Dataset에 역빈도 weight를 줘서 클래스 불균형
    # ('위험'/'주의' 표본이 '정상'보다 훨씬 적음)을 보정했었다. 여기서는 그 대신, 최종
    # 학습 세트(train_df)의 소수 클래스 "표본 자체"를 늘려서 균형을 맞추는 오버샘플링
    # (SMOTE)을 적용한다. 아래에서 만든 train_features_resampled/train_labels_resampled를
    # 다음 셀의 lgb.Dataset 생성에 그대로 사용한다(더 이상 weight는 쓰지 않는다).
    #
    # SMOTE 원리:
    #   1) 소수 클래스에 속한 표본 하나를 고른다.
    #   2) 같은 소수 클래스 안에서 그 표본의 k-최근접 이웃(KNN)을 찾는다.
    #   3) 표본과 이웃을 잇는 직선 위의 임의의 지점에 새로운 "합성" 표본을 만든다.
    #   => 원본을 그대로 복제(단순 중복)하는 게 아니라 새 값을 보간해서 만들기 때문에,
    #      단순 복제보다 모델이 소수 클래스를 외우기만 하는 과적합 위험이 적다.
    #
    # 왜 SMOTE가 아니라 SMOTENC인가:
    #   일반 SMOTE는 모든 피처가 연속형(숫자)이라고 가정하고 "두 값 사이를 보간"한다.
    #   하지만 우리 피처에는 issuetype_name/priority_name 같은 범주형(category) 컬럼이
    #   섞여 있어서, 이런 컬럼까지 숫자처럼 보간하면 존재하지도 않는 값이 생겨버린다.
    #   SMOTENC(SMOTE for Nominal and Continuous)는 연속형 피처는 SMOTE처럼 보간하고,
    #   범주형 피처는 이웃들의 값 중 하나를 그대로 가져오는 방식으로 처리해준다.
    #   그래서 어떤 컬럼이 범주형인지 인덱스로 알려줘야 한다.
    #

    # 학습 세트(train_df)에만 적용하고, 검증 세트(test_df)는 원본 분포 그대로 유지함:
    #   valid(test_df)는 "실제 운영에서 만날 데이터 분포"를 흉내내야 검증 점수를 믿을 수 있다.
    #   여기에 합성 표본이 섞이면 모델이 검증셋의 정보를 미리 엿본 것과 같은 효과(데이터 누수)가
    #   생겨 검증 점수가 실제보다 부풀려진다. 그래서 SMOTE는 train/valid를 나눈 "이후",
    #   train 쪽에만 적용한다.
    from imblearn.over_sampling import SMOTENC

    # feature_names 안에서 범주형 컬럼이 몇 번째 위치인지 인덱스로 변환 — SMOTENC는
    # 컬럼 이름이 아니라 인덱스(또는 불리언 마스크)로 범주형 위치를 지정받는다.
    categorical_feature_indices = [feature_names.index(c) for c in categorical_columns]

    # SMOTENC는 내부적으로 k-최근접 이웃 거리를 계산하므로 NaN을 다루지 못한다.
    # (예: 예상 소요시간이 없는 이슈는 progress_ratio_at_cutoff가 NaN — LightGBM은 이 NaN을
    # "결측치"로 그대로 학습에 활용하지만, SMOTENC에 넣기 전에는 임시로 0으로 채워야 한다.)
    # 범주형 컬럼은 0이 유효한 카테고리가 아닐 수 있으므로 숫자형 컬럼만 채운다.
    numeric_feature_names = [c for c in feature_names if c not in categorical_columns]
    train_features = train_df[feature_names].copy()
    train_features[numeric_feature_names] = train_features[numeric_feature_names].fillna(0)
    train_labels = train_df["risk_class"]

    print(f"SMOTE 적용 전 학습 세트 클래스 분포:\n{train_labels.value_counts().sort_index()}")

    # SMOTE의 기본 k_neighbors(5)는 "소수 클래스 표본 수 - 1"보다 클 수 없다(자기 자신을 뺀
    # 이웃 5개가 필요하므로 최소 6개 표본이 있어야 함). 표본이 아주 적은 경우를 대비해
    # k_neighbors를 소수 클래스 크기에 맞춰 동적으로 낮춘다.
    minority_class_count = train_labels.value_counts().min()
    if minority_class_count < 2:
        logger.warning(
            "소수 클래스 표본이 %d개뿐이라 SMOTE를 적용할 수 없어 원본 분포를 그대로 사용합니다.",
            minority_class_count,
        )
        train_features_resampled, train_labels_resampled = train_features, train_labels
    else:
        k_neighbors = min(5, minority_class_count - 1)
        smote = SMOTENC(
            categorical_features=categorical_feature_indices,
            k_neighbors=k_neighbors,
            random_state=42,
        )
        train_features_resampled, train_labels_resampled = smote.fit_resample(train_features, train_labels)

    print(f"\nSMOTE 적용 후 학습 세트 클래스 분포:\n{train_labels_resampled.value_counts().sort_index()}")
    print(f"학습 세트 행 수: {len(train_features)}행 -> {len(train_features_resampled)}행")


In [ ]:
if _RUN_TRAINING_CELLS:
    # 클래스 불균형은 위 셀의 SMOTE 오버샘플링으로 이미 해결했으므로(train_labels_resampled는
    # 세 클래스가 균등하다), 역빈도 가중치(weight) 없이 그대로 학습에 사용한다.
    train_set = lgb.Dataset(
        train_features_resampled,
        label=train_labels_resampled,
        categorical_feature=categorical_columns,
    )
    valid_set = lgb.Dataset(
        test_df[feature_names],
        label=test_df["risk_class"],
        categorical_feature=categorical_columns,
        reference=train_set,
    )

    booster = lgb.train(
        {
            "objective": "multiclass",
            "num_class": NUM_CLASSES,
            "metric": ["multi_logloss", "multi_error"],
            "learning_rate": 0.05,
            "num_leaves": 31,
            "verbose": -1,
        },
        train_set,
        num_boost_round=500,
        valid_sets=[train_set, valid_set],
        valid_names=["train", "valid"],
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )


In [ ]:
if _RUN_TRAINING_CELLS:
    # F1-Macro, classification report, 혼동 행렬로 클래스별 성능을 확인
    # (단순 accuracy는 '위험' 소수 클래스 성능을 가려서 부적합).
    probabilities = booster.predict(test_df[feature_names], num_iteration=booster.best_iteration)
    predicted_labels = np.argmax(probabilities, axis=1)

    class_labels = list(range(NUM_CLASSES))
    target_names = [RISK_CLASS_NAMES[i] for i in class_labels]

    print(
        "검증 F1-Macro: "
        f"{f1_score(test_df['risk_class'], predicted_labels, labels=class_labels, average='macro'):.4f}"
    )
    print(
        classification_report(
            test_df["risk_class"],
            predicted_labels,
            labels=class_labels,
            target_names=target_names,
            zero_division=0,
        )
    )

    cm = confusion_matrix(test_df["risk_class"], predicted_labels, labels=class_labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=target_names,
        yticklabels=target_names,
        cbar=False,
        ax=ax,
    )
    ax.set_xlabel("예측")
    ax.set_ylabel("실제")
    ax.set_title("혼동 행렬 (행=실제 / 열=예측)")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
if _RUN_TRAINING_CELLS:
    artifact = ModelArtifact(
        booster=booster,
        feature_names=feature_names,
        categorical_columns=categorical_columns,
        frequency_maps=frequency_maps,
        proxy_deadline_map=proxy_deadline_map,
        global_median_duration_hours=global_median,
    )
    _save_artifact(artifact)


In [ ]:
if _RUN_TRAINING_CELLS:
    # <학습 완료 후 생성된 ModelArtifact 객체 시각화>
    print(f"선정된 최종 피처 개수: {len(artifact.feature_names)}")
    display(artifact.feature_names)
    print(f"범주형 피처: {artifact.categorical_columns}")
    print(f"빈도 인코딩 대상 컬럼: {list(artifact.frequency_maps.keys())}")


## 8. 저장된 모델로 추론 테스트

In [ ]:
# 이 노트북 안에서는 load_artifact()가 이 노트북 자체의 전역변수 _artifact_cache를 참조하므로
# (위 5~6번 셀에서 함수를 그대로 재정의했기 때문), 방금 학습한 artifact를 바로 캐시에 넣어
# 재학습 없이 재사용할 수 있습니다.
if _RUN_TRAINING_CELLS:
    _artifact_cache = artifact

    # 테스트용 데이터
    sample_row = df.drop(columns=["risk_class", "created", "issue_key"]).iloc[99]  # id=99번인 이슈
    display(sample_row)

In [ ]:
if _RUN_TRAINING_CELLS:
    # '지연 위험도 예측' 테스트 실행
    probabilities = predict_class_probabilities(sample_row)
    class_probabilities = dict(zip(["정상", "주의", "위험"], probabilities))
    display(pd.Series(class_probabilities, name="확률"))

    predicted_class = max(class_probabilities, key=class_probabilities.get)
    print(f"예측 클래스: {predicted_class} (확률 {class_probabilities[predicted_class]:.4f})")
